O objetivo aqui é aplicar o método de bounding box nas mascaras binárias

In [1]:
import os 
import shutil
OUTPUT_F = "output_folder"
INPUT_F = "E:\Resultados experimentos\SAMU_SAO_CARLOS\controle_de_via_aerea_e_ventilacao\cam_traseira\mascaras_mediano_erosao\mascara_mediana_frame_2731.jpg"
 #esse método deve ser chamado para tratar erros de caminho
def verificar_diretorio(caminho):
    if os.path.exists(caminho):
        print("Caminho encontrado")
    else:
        print("ERRO: Caminho não encontrado")

def criar_output_f(caminho):
    if not os.path.exists(caminho):
        os.makedirs(caminho)
        print(f"Pasta '{caminho}' criada com sucesso.")
        
verificar_diretorio(INPUT_F)





Caminho encontrado


É preciso primeiro fechar os contornos das mascaras binárias, para isso será utilizado o algoritmo de convex hull


In [5]:
import cv2 as cv
import numpy as np

# Caminho da máscara binária
path = r"E:\Resultados experimentos\SAMU_SAO_CARLOS\controle_de_via_aerea_e_ventilacao\cam_traseira\mascaras_frameDiff_dilatacao\diff_frame_3931.jpg"

# Carregar imagem em escala de cinza
mask = cv.imread(path, cv.IMREAD_GRAYSCALE)

if mask is None:
    print("Erro ao carregar a imagem.")
    exit()

# Garantir que é binária
_, mask_bin = cv.threshold(mask, 127, 255, cv.THRESH_BINARY)

# Encontrar contornos
contours, _ = cv.findContours(mask_bin, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

#  FILTRO DE ÁREA
filtered_contours = []
for cnt in contours:
    area = cv.contourArea(cnt)
    if area > 2000:  # ajuste esse valor conforme necessário
        filtered_contours.append(cnt)

#  Convex hull dos contornos filtrados
hull_list = []
for cnt in filtered_contours:
    hull = cv.convexHull(cnt)
    hull_list.append(hull)

# Criar imagem para desenhar
drawing = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)

# Desenhar resultados
for i in range(len(filtered_contours)):
    # contorno (azul)
    cv.drawContours(drawing, filtered_contours, i, (255, 0, 0), 1)
    
    # convex hull (verde)
    cv.drawContours(drawing, hull_list, i, (0, 255, 0), 2)

# Mostrar
cv.imshow("Mascara Binaria", mask_bin)
cv.imshow("Convex Hull Filtrado", drawing)

cv.waitKey(0)
cv.destroyAllWindows()

Estratégia impregando o agrupamento de polignos convexos próximos:

In [ ]:
import cv2 as cv
import numpy as np

path = r"E:\Resultados experimentos\SAMU_SAO_CARLOS\controle_de_via_aerea_e_ventilacao\cam_traseira\mascaras_frameDiff_dilatacao\diff_frame_3931.jpg"

mask = cv.imread(path, cv.IMREAD_GRAYSCALE)
_, mask_bin = cv.threshold(mask, 127, 255, cv.THRESH_BINARY)

contours, _ = cv.findContours(mask_bin, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

#Filtrar por área
min_area = 2000
filtered = [cnt for cnt in contours if cv.contourArea(cnt) > min_area]

#Calcular centroides
centroids = []
for cnt in filtered:
    M = cv.moments(cnt)
    if M["m00"] != 0:
        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])
        centroids.append((cx, cy))
    else:
        centroids.append((0, 0))

# Agrupamento simples por proximidade
groups = []
distance_threshold = 500  # ajuste importante

for i, cnt in enumerate(filtered):
    placed = False

    for group in groups:
        for j in group:
            d = np.linalg.norm(np.array(centroids[i]) - np.array(centroids[j]))
            
            if d < distance_threshold:
                group.append(i)
                placed = True
                break
        
        if placed:
            break

    if not placed:
        groups.append([i])

#Criar imagem de saída
drawing = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)

#Processar cada grupo
for group in groups:
    group_contours = [filtered[i] for i in group]
    
    all_points = np.vstack(group_contours)
    
    x, y, w, h = cv.boundingRect(all_points)

    # desenhar bounding box
    cv.rectangle(drawing, (x, y), (x+w, y+h), (0, 255, 0), 2)

    # opcional: desenhar hull do grupo
    hull = cv.convexHull(all_points)
    cv.drawContours(drawing, [hull], -1, (0, 0, 255), 2)

# Debug: desenhar contornos originais
cv.drawContours(drawing, filtered, -1, (255, 0, 0), 1)

cv.imshow("Resultado Agrupado", drawing)
cv.imshow("Mascara", mask_bin)

cv.waitKey(0)
cv.destroyAllWindows()

Código em formato funcional iterando folders:

In [ ]:
import cv2 as cv
import numpy as np
import os

def processar_pasta_mascaras(input_dir, output_dir,
                             min_area=2000,
                             distance_threshold=500):

    #Criar pasta de saída se não existir
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    #Listar arquivos
    arquivos = [f for f in os.listdir(input_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    for nome_arquivo in arquivos:
        path = os.path.join(input_dir, nome_arquivo)

        mask = cv.imread(path, cv.IMREAD_GRAYSCALE)

        if mask is None:
            print(f"Erro ao carregar: {nome_arquivo}")
            continue

        #Binarização
        _, mask_bin = cv.threshold(mask, 127, 255, cv.THRESH_BINARY)

        #Contornos
        contours, _ = cv.findContours(mask_bin, cv.RETR_EXTERNAL, cv.CHAIN_APPROX_SIMPLE)

        #Filtro de área
        filtered = [cnt for cnt in contours if cv.contourArea(cnt) > min_area]

        #Centroides
        centroids = []
        for cnt in filtered:
            M = cv.moments(cnt)
            if M["m00"] != 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                centroids.append((cx, cy))
            else:
                centroids.append((0, 0))

        #Agrupamento
        groups = []

        for i, cnt in enumerate(filtered):
            placed = False

            for group in groups:
                for j in group:
                    d = np.linalg.norm(np.array(centroids[i]) - np.array(centroids[j]))

                    if d < distance_threshold:
                        group.append(i)
                        placed = True
                        break

                if placed:
                    break

            if not placed:
                groups.append([i])

        #Criar imagem de saída
        drawing = np.zeros((mask.shape[0], mask.shape[1], 3), dtype=np.uint8)

        #Processar grupos
        for group in groups:
            group_contours = [filtered[i] for i in group]

            all_points = np.vstack(group_contours)

            x, y, w, h = cv.boundingRect(all_points)

            # bounding box
            cv.rectangle(drawing, (x, y), (x+w, y+h), (0, 255, 0), 2)

            # convex hull do grupo
            hull = cv.convexHull(all_points)
            cv.drawContours(drawing, [hull], -1, (0, 0, 255), 2)

        #Debug: contornos
        cv.drawContours(drawing, filtered, -1, (255, 0, 0), 1)

        #Salvar imagem
        output_path = os.path.join(output_dir, nome_arquivo)
        cv.imwrite(output_path, drawing)

        print(f"Processado: {nome_arquivo}")

input_dir = r"E:\Resultados experimentos\SAMU_SAO_CARLOS\controle_de_via_aerea_e_ventilacao\cam_traseira\mascaras_frameDiff_dilatacao"
output_dir = os.path.join(input_dir, "resultados_bbox")

processar_pasta_mascaras(input_dir, output_dir)

Processado: diff_frame_0030.jpg
Processado: diff_frame_0060.jpg
Processado: diff_frame_0090.jpg
Processado: diff_frame_0120.jpg
Processado: diff_frame_0150.jpg
Processado: diff_frame_0180.jpg
Processado: diff_frame_0210.jpg
Processado: diff_frame_0240.jpg
Processado: diff_frame_0270.jpg
Processado: diff_frame_0300.jpg
Processado: diff_frame_0330.jpg
Processado: diff_frame_0360.jpg
Processado: diff_frame_0390.jpg
Processado: diff_frame_0420.jpg
Processado: diff_frame_0450.jpg
Processado: diff_frame_0480.jpg
Processado: diff_frame_0510.jpg
Processado: diff_frame_0540.jpg
Processado: diff_frame_0570.jpg
Processado: diff_frame_0600.jpg
Processado: diff_frame_0630.jpg
Processado: diff_frame_0660.jpg
Processado: diff_frame_0690.jpg
Processado: diff_frame_0720.jpg
Processado: diff_frame_0750.jpg
Processado: diff_frame_0780.jpg
Processado: diff_frame_0810.jpg
Processado: diff_frame_0840.jpg
Processado: diff_frame_0870.jpg
Processado: diff_frame_0900.jpg
Processado: diff_frame_0930.jpg
Processa

O bloco de código abaixo realiza a bbox sob o Hull dos centroids:

In [ ]:
import cv2 as cv
import numpy as np

path = r"E:\Resultados experimentos\SAMU_SAO_CARLOS\controle_de_via_aerea_e_ventilacao\cam_traseira\mascaras_frameDiff_dilatacao\diff_frame_3931.jpg"

# =========================
# Leitura da máscara
# =========================

mask = cv.imread(path, cv.IMREAD_GRAYSCALE)

_, mask_bin = cv.threshold(
    mask,
    127,
    255,
    cv.THRESH_BINARY
)

# =========================
# Encontrar contornos
# =========================

contours, _ = cv.findContours(
    mask_bin,
    cv.RETR_EXTERNAL,
    cv.CHAIN_APPROX_SIMPLE
)

# =========================
# Filtrar por área
# =========================

min_area = 2000

filtered = [
    cnt for cnt in contours
    if cv.contourArea(cnt) > min_area
]

# =========================
# Calcular centroides
# =========================

centroids = []

for cnt in filtered:

    M = cv.moments(cnt)

    if M["m00"] != 0:

        cx = int(M["m10"] / M["m00"])
        cy = int(M["m01"] / M["m00"])

        centroids.append((cx, cy))

    else:

        centroids.append((0, 0))

# =========================
# Agrupamento por proximidade
# =========================

groups = []

distance_threshold = 500

for i, cnt in enumerate(filtered):

    placed = False

    for group in groups:

        for j in group:

            d = np.linalg.norm(
                np.array(centroids[i]) -
                np.array(centroids[j])
            )

            if d < distance_threshold:

                group.append(i)

                placed = True

                break

        if placed:
            break

    if not placed:
        groups.append([i])

# =========================
# Criar imagem de saída
# =========================

drawing = np.zeros(
    (mask.shape[0], mask.shape[1], 3),
    dtype=np.uint8
)

# =========================
# Processar grupos
# =========================

for group in groups:

    # -------------------------
    # Obter centroides do grupo
    # -------------------------

    points = np.array(
        [centroids[i] for i in group],
        dtype=np.int32
    )

    # OpenCV espera formato:
    # (N,1,2)

    points = points.reshape((-1, 1, 2))

    # -------------------------
    # Criar convex hull
    # -------------------------

    if len(points) >= 3:

        hull = cv.convexHull(points)

    else:

        hull = points

    # -------------------------
    # Bounding box do hull
    # -------------------------

    x, y, w, h = cv.boundingRect(hull)

    # -------------------------
    # Desenhar hull
    # -------------------------

    cv.drawContours(
        drawing,
        [hull],
        -1,
        (0, 0, 255),
        2
    )

    # -------------------------
    # Desenhar bounding box
    # -------------------------

    cv.rectangle(
        drawing,
        (x, y),
        (x + w, y + h),
        (0, 255, 0),
        2
    )

    # -------------------------
    # Desenhar centroides
    # -------------------------

    for p in points:

        cx, cy = p[0]

        cv.circle(
            drawing,
            (cx, cy),
            5,
            (255, 255, 0),
            -1
        )

# =========================
# Debug:
# Contornos originais
# =========================

cv.drawContours(
    drawing,
    filtered,
    -1,
    (255, 0, 0),
    1
)

# =========================
# Mostrar resultado
# =========================

cv.imshow("Resultado Agrupado", drawing)

cv.imshow("Mascara", mask_bin)

cv.waitKey(0)

cv.destroyAllWindows()
